<center>
    <h1>Probabilistic models taken by Storm</h1>
    <h2>Step 4: MDPs with more uncertainty</h2>
    <br>
    <br>
    <img src="images/storm_logo.png" width="500px"/>
    <h3>Matthias Volk</h3>
</center>

<div align="right">Press <em>spacebar</em> to navigate</div>


# Preparation
Load models from before

In [ ]:
# Import prism model from the previous step
import stormvogel
import stormpy

from copy import deepcopy
from examples.orchard_game_stormvogel import Orchard, available_actions, delta, labels, rewards, Fruit, GameState, DiceOutcome
from examples.orchard_builder import build_simple

In [ ]:
# Skip

# Interval MDP
- allow uncertainty in dice role
- point estimates $\frac{1}{6}$ become intervals $[\frac{5}{36}, \frac{7}{36}]$

In [ ]:
# The transition function
def delta(state, action):
    if state.game_state() != GameState.NOT_ENDED:
        # Game has ended -> self loop
        return [(1, state)]

    if state.dice is None:
        # Player throws dice and considers outcomes
        outcomes = []
        # Probability of fair dice throw over
        # each fruit type + 1 basket + 1 raven
        fair_dice_prob = 1 / (len(state.trees.keys()) + 2)
        # NEW: adapted probability
        fair_dice_prob = stormvogel.model.Interval(fair_dice_prob-(1/36), fair_dice_prob+(1/36))

        # 1. Dice shows fruit
        for fruit in state.trees.keys():
            next_state = deepcopy(state)
            next_state.dice = DiceOutcome.FRUIT, fruit
            outcomes.append((fair_dice_prob, next_state))

        # 2. Dice shows basket
        next_state = deepcopy(state)
        next_state.dice = DiceOutcome.BASKET, None
        outcomes.append((fair_dice_prob, next_state))

        # 3. Dice shows raven
        next_state = deepcopy(state)
        next_state.dice = DiceOutcome.RAVEN, None
        outcomes.append((fair_dice_prob, next_state))
        return outcomes

    elif state.dice[0] == DiceOutcome.FRUIT:
        # Player picks specified fruit
        fruit = state.dice[1]
        next_state = deepcopy(state)
        next_state.pick_fruit(fruit)
        next_state.next_round()
        return [(1, next_state)]

    elif state.dice[0] == DiceOutcome.BASKET:
        assert action.startswith("choose")
        # Player chooses fruit specified by action
        fruit = Fruit[action.removeprefix("choose")]
        next_state = deepcopy(state)
        next_state.pick_fruit(fruit)
        next_state.next_round()
        return [(1, next_state)]

    elif state.dice[0] == DiceOutcome.RAVEN:
        next_state = deepcopy(state)
        next_state.move_raven()
        next_state.next_round()
        return [(1, next_state)]

    assert False

In [ ]:
# skip

## Build interval MDP
- build model as before and obtain interval MDP

In [ ]:
init_game = Orchard([Fruit.APPLE, Fruit.CHERRY, Fruit.PEAR, Fruit.PLUM],
                    num_fruits=4,
                    raven_distance=5)
    
# For the full model, we only set the relevant labels for the winning conditions
# and do not expose the internal state information
def labels_full(state):
    labels = []
    if state.game_state() == GameState.PLAYERS_WON:
        labels.append("PlayersWon")
    elif state.game_state() == GameState.RAVEN_WON:
        labels.append("RavenWon")
    return labels

orchard = stormvogel.bird.build_bird(
    modeltype=stormvogel.ModelType.MDP,
    init=init_game,
    available_actions=available_actions,
    delta=delta,
    labels=labels_full,
    max_size=100000
)

# Convert to stormpy model
orchard_storm = stormvogel.mapping.stormvogel_to_stormpy(orchard)

In [ ]:
print(orchard_storm)

In [ ]:
# Skip

## Analyse interval MDP
- analyse w.r.t. robust or cooperative resolution mode

In [ ]:
# Parse properties
properties = stormpy.parse_properties('Pmax=? [F "PlayersWon"]')
task = stormpy.CheckTask(properties[0].raw_formula)
# Prepare model checking
env = stormpy.Environment()
# Set VI algorithm to plain VI
env.solver_environment.minmax_solver_environment.method = stormpy.MinMaxMethod.value_iteration

In [ ]:
# Set resolution mode
task.set_uncertainty_resolution_mode(
    stormpy.UncertaintyResolutionMode.ROBUST
)
# Check model
stormpy_result = stormpy.check_interval_mdp(orchard_storm, task, env)
print(stormpy_result.at(orchard_storm.initial_states[0]))

In [ ]:
# skip

# Parametric MDP
- in interval MDP: can choose different dice probabilities per throw
- more realistic: use uncertain but fixed probability for all throws
- use parameters in MDP:
  - parameter `p` is probability to roll 🍏 or 🍐 *(only two fruits now)*
  - parameter `q` is probability to roll 🧺
  - probability to roll 🐦‍⬛ is `1−2p−q`

## Build parametric MDP
- create polynomials for transition probabilities
- adapt transition function

In [ ]:
# Create the correct polynomials
one = stormvogel.parametric.Polynomial([])
one.add_term((0,), 1)

params = [f"p{i}" for i in range(2)]

parameters = [stormvogel.parametric.Polynomial([p]) for p in params]
for i in range(2):
    parameters[i].add_term((1,), 1)

parameters.append(stormvogel.parametric.Polynomial(params))
parameters[-1].add_term((0, 1), -1)
parameters[-1].add_term((1, 0), -2)
parameters[-1].add_term((0, 0), 1)

In [ ]:
# Change delta function
def delta_pmc(state, action):
    if state.game_state() != GameState.NOT_ENDED:
        # Game has ended -> self loop
        return [(1, state)]

    if state.dice is None:
        # Player throws dice and considers outcomes
        outcomes = []
        # Total outcomes is number of fruits + 1 for basket + 1 for raven
        total_outcomes = (len(state.trees.keys()) + 2)
        assert total_outcomes == 4

        i = 0
        # 1. Dice shows fruit
        for fruit in state.trees.keys():
            next_state = deepcopy(state)
            next_state.dice = DiceOutcome.FRUIT, fruit
            # NEW: parametric probabilities
            outcomes.append((parameters[i], next_state))
        i += 1
            
        # 2. Dice shows basket
        next_state = deepcopy(state)
        next_state.dice = DiceOutcome.BASKET, 0
        # NEW: parametric probabilities
        outcomes.append((parameters[i], next_state))
       
        # 3. Dice shows raven
        next_state = deepcopy(state)
        next_state.dice = DiceOutcome.RAVEN, None
        # NEW: parametric probabilities
        outcomes.append((parameters[i+1], next_state))
        assert len(outcomes) == total_outcomes
        return outcomes

    elif state.dice[0] == DiceOutcome.FRUIT:
        # Player picks specified fruit
        fruit = state.dice[1]
        next_state = deepcopy(state)
        next_state.pick_fruit(fruit)
        next_state.next_round()
        return [(one, next_state)]

    elif state.dice[0] == DiceOutcome.BASKET:
        assert action.startswith("choose")
        # Player chooses fruit specified by action
        fruit = Fruit[action.removeprefix("choose")]
        next_state = deepcopy(state)
        next_state.pick_fruit(fruit)
        next_state.next_round()
        return [(one, next_state)]

    elif state.dice[0] == DiceOutcome.RAVEN:
        next_state = deepcopy(state)
        next_state.move_raven()
        next_state.next_round()
        return [(one, next_state)]
        
    assert False

In [ ]:
# Skip

In [ ]:
init_small = Orchard([Fruit.APPLE, Fruit.CHERRY],
                     num_fruits=2,
                     raven_distance=2)
orchard_pmc = stormvogel.bird.build_bird(
    modeltype=stormvogel.ModelType.MDP,
    init=init_small,
    available_actions=available_actions,
    delta=delta_pmc,
    labels=labels_full,
    max_size=100000
)

# Convert to stormpy model
orchard_stormpy_pmdp = stormvogel.mapping.stormvogel_to_stormpy(orchard_pmc)
print(orchard_stormpy_pmdp)

In [ ]:
# skip

## Analysis of parametric MDP
1. apply optimal policy from non-parametric MDP on parametric MDP
2. yields parametric DTMC
3. compute solution function on parametric DTMC

In [ ]:
# Get scheduler from MDP
orchard_mdp_stormpy = stormvogel.mapping.stormvogel_to_stormpy(build_simple())
result = stormpy.model_checking(orchard_mdp_stormpy, properties[0].raw_formula, extract_scheduler=True)

# Apply scheduler on pMDP
scheduler = result.scheduler.cast_to_parametric_datatype()
orchard_stormpy_pmc = orchard_stormpy_pmdp.apply_scheduler(scheduler)

In [ ]:
# Skip

In [ ]:
# Perform model checking
properties2 = stormpy.parse_properties(
    'P=? [F "PlayersWon"]'
)
stormpy_result = stormpy.model_checking(orchard_stormpy_pmc, properties2[0].raw_formula)
solution_function = stormpy_result.at(0)
solution_function = str(solution_function).replace('_r_1', 'q').replace('_r_2', 'p')
print(solution_function)

![Plot of solution function](images/plot_pmc.png)

# Partially observable MDP
- adapted model and added observations (`hasApple`, ...)

In [ ]:
import stormpy
import stormpy.pomdp

prism_program = stormpy.parse_prism_program("examples/orchard_pomdp.pm")
formula_str = 'Pmax=? [!"RavenWon" U "PlayersWon"]'
properties = stormpy.parse_properties_for_prism_program(formula_str, prism_program)
prism_program, properties = stormpy.preprocess_symbolic_input(prism_program, properties, "")
prism_program = prism_program.as_prism_program()
options = stormpy.BuilderOptions([p.raw_formula for p in properties])
options.set_build_state_valuations()
options.set_build_choice_labels()
pomdp = stormpy.build_model(prism_program, properties)
pomdp = stormpy.pomdp.make_canonic(pomdp)
print(pomdp)

In [ ]:
# skip

## Analysis of POMDP

In [ ]:
belexpl_options = stormpy.pomdp.BeliefExplorationModelCheckerOptionsDouble(True, True)
belexpl_options.use_clipping = False
belexpl_options.refine = True

belmc = stormpy.pomdp.BeliefExplorationModelCheckerDouble(pomdp, belexpl_options)
result = belmc.check(properties[0].raw_formula, [])
print(f"Result in: [{result.lower_bound}, {result.upper_bound}]")

- Tight result, same as for perfect information
- Precision set to $10^{-6}$

## Analysis of POMDP

- modify game:
  - randomly steal some fruit before the start<br>
    $\to$ starting conditions not perfectly clear anymore

In [ ]:
# Load model with stealing
prism_program = stormpy.parse_prism_program("examples/orchard_pomdp_steal.pm")
formula_str = 'Pmax=? [!"RavenWon" U "PlayersWon"]'
properties = stormpy.parse_properties_for_prism_program(formula_str, prism_program)
prism_program, properties = stormpy.preprocess_symbolic_input(prism_program, properties, "")
prism_program = prism_program.as_prism_program()
options = stormpy.BuilderOptions([p.raw_formula for p in properties])
options.set_build_state_valuations()
options.set_build_choice_labels()
pomdp_steal = stormpy.build_model(prism_program, properties)
pomdp_steal = stormpy.pomdp.make_canonic(pomdp_steal)
print(pomdp_steal)

In [ ]:
# skip

## Analysis of POMDP

In [ ]:
# Fully observable
mdp_res = stormpy.model_checking(pomdp_steal, properties[0], force_fully_observable=True)
print(mdp_res.at(pomdp_steal.initial_states[0]))

In [ ]:
belexpl_options = stormpy.pomdp.BeliefExplorationModelCheckerOptionsDouble(True, True)
belexpl_options.use_clipping = False
belexpl_options.refine = True

belmc = stormpy.pomdp.BeliefExplorationModelCheckerDouble(pomdp_steal, belexpl_options)
result = belmc.check(properties[0].raw_formula, [])
print(f"Result in: [{result.lower_bound}, {result.upper_bound}]")

In [ ]:
# skip